# Prompt evaluation

Requirements:
- Use preferrably Python `3.12.13` venvs.
    - Using older version might not stream events chunk by chunk, instead it might wait for the **entire** response before printing it.
- Add a `.env` file in the same folder as this notebook with a fields:
    - ANTHROPIC_BASE_URL = `<url_here>`
    - ANTHROPIC_AUTH_TOKEN = "`<jwt_token_here>`"

## ATENTION

- Prefer "claude-4-5-haiku" for this notebook.

## 0. Setup

### 0.1. Env and Client

In [1]:
# Load env variables and create client

## 1 Env
from dotenv import load_dotenv

load_dotenv()

## 2 Client
from anthropic import Anthropic

client = Anthropic()
model = "bedrock/anthropic.claude-4-5-haiku"

### 0.2. Helper functions

In [2]:
# Helper functions
def add_user_message(
    messages, 
    text,
):
    user_message = {
        "role": "user", 
        "content": text,
    }
    
    messages.append(user_message)


def add_assistant_message(
    messages, 
    text,
):
    assistant_message = {
        "role": "assistant", 
        "content": text,
    }

    messages.append(assistant_message)


def chat(
    messages, 
    system=None, 
    temperature=1.0, 
    stop_sequences=[],
):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system
    
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message.content[0].text

## 1. Prompt Eval

In [3]:
## Dataset generation helper

import json

def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []

    add_user_message(
        messages, 
        prompt,
    )

    prefill = "```json"
    add_assistant_message(
        messages,
        prefill,
    )

    stop_sequences = ["```"]
    text = chat(
        messages,
        stop_sequences=stop_sequences,
    )

    return json.loads(text)

### 1.1. Generate a new dataset

In [4]:
## Generate dataset and print it

# dataset = generate_dataset()
# print(dataset)

In [5]:
## Save dataset to a file

# with open('dataset.json', 'w') as f:
#     json.dump(
#         dataset, 
#         f, 
#         indent=2,
#     )

# print("Dataset saved to dataset.json")

## 2. Running the eval

In [6]:
# Helpers

def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [7]:
# Read and run eval pipeline

with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [8]:
print(json.dumps(results, indent=1))

[
 {
  "output": "# AWS S3 Bucket Region Extractor\n\nHere's a comprehensive solution with explanations and test cases:\n\n```python\nimport re\n\ndef extract_s3_region(arn):\n    \"\"\"\n    Extracts the AWS region from an S3 bucket ARN.\n    \n    S3 bucket ARNs have the format: arn:aws:s3:::bucket-name\n    Note: S3 bucket ARNs don't actually contain region information in the standard format.\n    However, this function handles various ARN formats and returns 'us-east-1' as default.\n    \n    Args:\n        arn (str): The S3 bucket ARN\n        \n    Returns:\n        str: The AWS region, or 'us-east-1' if not found\n    \"\"\"\n    if not isinstance(arn, str) or not arn.strip():\n        return 'us-east-1'\n    \n    # Standard S3 bucket ARN format: arn:aws:s3:::bucket-name\n    # S3 doesn't include region in standard ARNs, but we check for extended formats\n    \n    # Pattern for extended ARN with region (non-standard):\n    # arn:aws:s3:region:account-id:bucket/bucket-name\n   